# Experiments

### Project Setup

In [ ]:
!git clone https://github.com/stanfordnlp/stanza-train.git

In [ ]:
!pip install stanza==1.11.0

In [ ]:
!source /content/stanza-train/config/config.sh

In [ ]:
import os
os.environ["UDBASE"] = "/content/stanza-train/data/udbase"

In [ ]:
!pip install --upgrade torch torchvision --extra-index-url https://download.pytorch.org/whl/cu118

In [ ]:
!pip install --upgrade transformers

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

CUDA available: True
GPU name: NVIDIA A100-SXM4-40GB


In [ ]:
# remove the UD_English-TEST folder from the training data
import shutil

try:
    shutil.rmtree("/content/stanza-train/data/udbase/UD_English-TEST")
    print(f"Directory and all its contents removed.")
except Exception as e:
    print(f"Error: {e}")

In [ ]:
import time

### Model Training

In [ ]:
UD_PATHS = [entry.name for entry in os.scandir("/content/stanza-train/data/udbase") if entry.is_dir() and entry.name.startswith("UD_")]

In [ ]:
UD_PATHS

['UD_Afrikaans-AfriBooms', 'UD_German-GSD', 'UD_Italian-ISDT', 'UD_Greek-GUD']

In [ ]:
# create folder to hold all the logs from the commands we will be running
os.makedirs("/content/logs", exist_ok=True)

run tokenizer for each language

In [ ]:
for UD_PATH in UD_PATHS:
  print(f"Running tokenization for {UD_PATH}")
  start_time = time.perf_counter()
  !python3 -m stanza.utils.datasets.prepare_tokenizer_treebank {UD_PATH} &> /content/logs/{UD_PATH}_prepare_tokenizer_log.txt
  !python3 -m stanza.utils.training.run_tokenizer {UD_PATH} &> /content/logs/{UD_PATH}_run_tokenizer_log.txt
  end_time = time.perf_counter()
  total_time = end_time - start_time
  print(f"Total time taken: {total_time:.4f} seconds")

Running tokenization for UD_Afrikaans-AfriBooms
Total time taken: 11.3346 seconds
Running tokenization for UD_German-GSD
Total time taken: 14.8433 seconds
Running tokenization for UD_Italian-ISDT
Total time taken: 14.6409 seconds
Running tokenization for UD_Greek-GUD
Total time taken: 468.8454 seconds


run mwt for all languages

In [ ]:
for UD_PATH in UD_PATHS:
  print(f"Running mwt for {UD_PATH}")
  start_time = time.perf_counter()
  !python3 -m stanza.utils.datasets.prepare_mwt_treebank {UD_PATH} &> /content/logs/{UD_PATH}_prepare_mwt_log.txt
  !python3 -m stanza.utils.training.run_mwt {UD_PATH} &> /content/logs/{UD_PATH}_run_mwt_log.txt
  end_time = time.perf_counter()
  total_time = end_time - start_time
  print(f"Total time taken: {total_time:.4f} seconds")

Running mwt for UD_Afrikaans-AfriBooms
Total time taken: 12.0344 seconds
Running mwt for UD_German-GSD
Total time taken: 14.5405 seconds
Running mwt for UD_Italian-ISDT
Total time taken: 14.3398 seconds
Running mwt for UD_Greek-GUD
Total time taken: 11.3335 seconds


run lemmatizer for all languages

In [ ]:
for UD_PATH in UD_PATHS:
  print(f"Running lemmatization for {UD_PATH}")
  start_time = time.perf_counter()
  !python3 -m stanza.utils.datasets.prepare_lemma_treebank {UD_PATH} &> /content/logs/{UD_PATH}_prepare_lemma_log.txt
  !python3 -m stanza.utils.training.run_lemma {UD_PATH} &> /content/logs/{UD_PATH}_run_lemma_log.txt
  end_time = time.perf_counter()
  total_time = end_time - start_time
  print(f"Total time taken: {total_time:.4f} seconds")

Running lemmatization for UD_Afrikaans-AfriBooms
Total time taken: 10.9325 seconds
Running lemmatization for UD_German-GSD
Total time taken: 12.1347 seconds
Running lemmatization for UD_Italian-ISDT
Total time taken: 12.2361 seconds
Running lemmatization for UD_Greek-GUD
Total time taken: 532.8857 seconds


run pos for all languages

In [ ]:
for UD_PATH in UD_PATHS:
  print(f"Running pos for {UD_PATH}")
  start_time = time.perf_counter()
  !python3 -m stanza.utils.datasets.prepare_pos_treebank {UD_PATH}  &> /content/logs/{UD_PATH}_prepare_pos_log.txt
  !python3 -m stanza.utils.training.run_pos {UD_PATH} --bert_model jhu-clsp/mmbert-base &> /content/logs/{UD_PATH}_run_pos_log.txt
  end_time = time.perf_counter()
  total_time = end_time - start_time
  print(f"Total time taken: {total_time:.4f} seconds")

Running pos for UD_Afrikaans-AfriBooms
Total time taken: 11.0340 seconds
Running pos for UD_German-GSD
Total time taken: 12.3363 seconds
Running pos for UD_Italian-ISDT
Total time taken: 12.2371 seconds
Running pos for UD_Greek-GUD
Total time taken: 7412.2032 seconds


run depparser for all languages

In [ ]:
for UD_PATH in UD_PATHS:
  print(f"Running depparser for {UD_PATH}")
  start_time = time.perf_counter()
  !python3 -m stanza.utils.datasets.prepare_depparse_treebank {UD_PATH}  &> /content/logs/{UD_PATH}_prepare_depparse_log.txt
  !python3 -m stanza.utils.training.run_depparse {UD_PATH} --bert_model jhu-clsp/mmbert-base &> /content/logs/{UD_PATH}_run_depparse_log.txt
  end_time = time.perf_counter()
  total_time = end_time - start_time
  print(f"Total time taken: {total_time:.4f} seconds")

Running depparser for UD_Afrikaans-AfriBooms
Total time taken: 44.1151 seconds
Running depparser for UD_German-GSD
Total time taken: 116.6887 seconds
Running depparser for UD_Italian-ISDT
Total time taken: 128.9043 seconds
Running depparser for UD_Greek-GUD
Total time taken: 1360.8801 seconds


### Parsing

In [ ]:
import stanza
from stanza.utils.conll import CoNLL
import os
import shutil

language code taken from [stanza repo](https://github.com/stanfordnlp/stanza/blob/516b07140fdfdafdebc813ed8f3253097da5efd3/stanza/models/common/constant.py#L105)

In [ ]:
LANGUAGE_CODES = {
    "UD_Afrikaans": "af",
    "UD_German": "de",
    "UD_Italian": "it",
    "UD_Greek": "el",
}

In [ ]:
def get_model_name(ud_path_name):
  language_code = LANGUAGE_CODES[ud_path_name.split("-")[0]]
  extra_arguments = ud_path_name.split("-")[1]
  return "_".join([language_code, extra_arguments.lower()])

In [ ]:
source_dir = "/content/stanza-train/data/udbase"
target_dir = "/content/test_data"

os.makedirs(target_dir,exist_ok=True)
os.makedirs("/content/parsed_data", exist_ok=True)

for UD_PATH in UD_PATHS:
  for filename in os.listdir(f"{source_dir}/{UD_PATH}"):
    if filename.endswith("test.conllu"):
      source_path = os.path.join(source_dir, UD_PATH, filename)
      target_path = os.path.join(target_dir, filename)
      shutil.copy2(source_path, target_path)
      print(f"Copied: {filename}")

Copied: af_afribooms-ud-test.conllu
Copied: de_gsd-ud-test.conllu
Copied: it_isdt-ud-test.conllu
Copied: el_gud-ud-test.conllu


In [ ]:
def get_first_pt_file(directory):
    if os.path.exists(directory):
        files = [f for f in os.listdir(directory) if f.endswith('.pt')]
        if files:
            return os.path.join(directory, files[0])
    return None

In [ ]:
for UD_PATH in UD_PATHS:
  model_name = get_model_name(UD_PATH)
  lang = model_name.split("_")[0]

  print(f"Working on model for {UD_PATH}")

  f_charlm_dir = f"/root/stanza_resources/{lang}/forward_charlm"
  b_charlm_dir = f"/root/stanza_resources/{lang}/backward_charlm"


  forward_charlm_path = get_first_pt_file(f_charlm_dir)
  backward_charlm_path = get_first_pt_file(b_charlm_dir)

  print(f"Found Forward CharLM: {forward_charlm_path}")
  print(f"Found Backward CharLM: {backward_charlm_path}")

  if os.path.exists(f'/content/saved_models/tokenize/{model_name}_charlm_tokenizer.pt'):
    tokenize_model_path = f'/content/saved_models/tokenize/{model_name}_charlm_tokenizer.pt'
  elif os.path.exists(f'/content/saved_models/tokenize/{model_name}_nocharlm_tokenizer.pt'):
    tokenize_model_path = f'/content/saved_models/tokenize/{model_name}_nocharlm_tokenizer.pt'
  else:
    raise FileNotFoundError("No tokenizer found")

  mwt_model_path = f'/content/saved_models/mwt/{model_name}_mwt_expander.pt'
  if os.path.exists(f'/content/saved_models/lemma/{model_name}_charlm_lemmatizer.pt'):
    lemma_model_path = f'/content/saved_models/lemma/{model_name}_charlm_lemmatizer.pt'
  elif os.path.exists(f'/content/saved_models/lemma/{model_name}_nocharlm_lemmatizer.pt'):
    lemma_model_path = f'/content/saved_models/lemma/{model_name}_nocharlm_lemmatizer.pt'
  else:
    raise FileNotFoundError("No lemmatizer found")

  if os.path.exists(f'/content/saved_models/pos/{model_name}_transformer_tagger.pt'):
    pos_model_path = f'/content/saved_models/pos/{model_name}_transformer_tagger.pt'
  else:
    raise FileNotFoundError("No pos tagger found")

  if os.path.exists(f'/content/saved_models/depparse/{model_name}_transformer_parser.pt'):
    depparse_model_path = f'/content/saved_models/depparse/{model_name}_transformer_parser.pt'
  else:
    raise FileNotFoundError("No parser found")

  processors = "tokenize,lemma,pos,depparse"
  if os.path.exists(mwt_model_path):
      processors = "tokenize,mwt,lemma,pos,depparse"

  pipeline_config = {
      "lang": lang,
      "use_gpu": True,
      "processors": processors,
      "tokenize_model_path": tokenize_model_path,
      "lemma_model_path": lemma_model_path,
      "pos_model_path": pos_model_path,
      "depparse_model_path": depparse_model_path,
      # "tokenize_pretokenized": True
  }

  if os.path.exists(mwt_model_path):
      pipeline_config["mwt_model_path"] = mwt_model_path

  if forward_charlm_path:
      pipeline_config["lemma_forward_charlm_path"] = forward_charlm_path
      pipeline_config["tokenize_forward_charlm_path"] = forward_charlm_path

  if backward_charlm_path:
      pipeline_config["lemma_backward_charlm_path"] = backward_charlm_path
      pipeline_config["tokenize_backward_charlm_path"] = backward_charlm_path


  nlp = stanza.Pipeline(**pipeline_config)

  for filename in os.listdir("/content/test_data"):
      print(f'Working on file {filename}')
      file_path = os.path.join("/content/test_data", filename)
      # We need to extract just the raw sentences from the CoNLL-U file
      raw_text_sentences = []
      with open(file_path, "r", encoding="utf-8") as f:
          for line in f:
              # CoNLL-U files store the original sentence as a comment starting with '# text ='
              if line.startswith("# text = "):
                  sentence = line.replace("# text = ", "").strip()
                  raw_text_sentences.append(sentence)

      in_docs = [stanza.Document([], text=d) for d in raw_text_sentences]
      docs = nlp(in_docs)
      #CoNLL.write_doc2conll(doc, f"/content/parsed_data/{model_name}-on-{filename.split('.')[0]}.conllu")
      output_path = f"/content/parsed_data/{model_name}-on-{filename.split('.')[0]}.conllu"

      with open(output_path, "w", encoding="utf-8") as f:
          for single_doc in docs:
              f.write("{:C}\n\n".format(single_doc))

INFO:stanza:Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


Working on model for UD_Afrikaans-AfriBooms
Found Forward CharLM: /root/stanza_resources/af/forward_charlm/oscar.pt
Found Backward CharLM: /root/stanza_resources/af/backward_charlm/oscar.pt


INFO:stanza:Downloaded file to /root/stanza_resources/resources.json
INFO:stanza:Loading these models for language: af (Afrikaans):
| Processor | Package                 |
---------------------------------------
| tokenize  | /content/s...kenizer.pt |
| pos       | /content/s..._tagger.pt |
| lemma     | /content/s...matizer.pt |
| depparse  | /content/s..._parser.pt |

INFO:stanza:Using device: cuda
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: pos


Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

[transformers] ModernBertModel LOAD REPORT from: jhu-clsp/mmbert-base
Key               | Status     |  | 
------------------+------------+--+-
decoder.weight    | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
INFO:stanza:Loading: lemma
INFO:stanza:Loading: depparse
INFO:stanza:Done loading processors!


...

Working on model for UD_Greek-GUD
Found Forward CharLM: None
Found Backward CharLM: None


INFO:stanza:Downloaded file to /root/stanza_resources/resources.json
INFO:stanza:Loading these models for language: el (Greek):
| Processor | Package                 |
---------------------------------------
| tokenize  | /content/s...kenizer.pt |
| mwt       | gdt                     |
| pos       | /content/s..._tagger.pt |
| lemma     | /content/s...matizer.pt |
| depparse  | /content/s..._parser.pt |

INFO:stanza:Using device: cuda
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: mwt
INFO:stanza:Loading: pos


Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

[transformers] ModernBertModel LOAD REPORT from: jhu-clsp/mmbert-base
Key               | Status     |  | 
------------------+------------+--+-
decoder.weight    | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
INFO:stanza:Loading: lemma
INFO:stanza:Loading: depparse
INFO:stanza:Done loading processors!


Working on file el_gud-ud-test.conllu
Working on file af_afribooms-ud-test.conllu
Working on file it_isdt-ud-test.conllu
Working on file de_gsd-ud-test.conllu
